In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

In [21]:
df = pd.read_csv('data/WESAD_S2_features.csv')
print(df.shape)
display(df.head())

(11973, 66)


,Time,MEAN_RR,MEDIAN_RR,SDRR,RMSSD,SDSD,SDRR_RMSSD,HR,pNN25,pNN50,...,KURT_SQUARE,HR_SQRT,MEAN_RR_MEAN_MEAN_REL_RR,SD2_LF,HR_LF,HR_HF,HF_VLF,subject id,condition,SSSQ
0,7.083262,897.392513,873.562810,170.971816,19.538659,19.493030,8.750437,68.865469,13.25,1.5,...,9.114545,8.298522,703684.111546,0.174063,0.049657,1.965697,0.008327,2,baseline,5
1,7.087428,896.005887,873.513830,169.558513,19.567190,19.516526,8.665450,68.943400,13.25,1.5,...,9.944790,8.303216,666317.132437,0.171156,0.049291,1.935464,0.008348,2,baseline,5
2,7.091595,894.600572,872.908040,168.076589,19.567218,19.516423,8.589703,69.022050,13.25,1.5,...,10.840055,8.307951,669170.553087,0.168061,0.048884,1.902749,0.008368,2,baseline,5
3,7.095762,893.232563,872.211295,166.548000,19.579760,19.532607,8.506131,69.097187,13.25,1.5,...,11.819928,8.312472,700739.775450,0.164809,0.048432,1.867660,0.008392,2,baseline,5
4,7.099928,891.925820,871.400185,165.009231,19.599071,19.556585,8.419237,69.167573,13.25,1.5,...,12.895677,8.316704,742959.637571,0.161443,0.047936,1.830138,0.008422,2,baseline,5


In [22]:
# Core signals
features = ['HR', 'RMSSD', 'SDRR', 'MEAN_RR']

# Missing values and summary
print("Condition breakdown:")
print(df['condition'].value_counts())

print("\nSummary stats for primary signals:")
display(df[features].describe())

Condition breakdown:
condition
baseline      4617
meditation    3284
stress        2580
amusement     1492
Name: count, dtype: int64

Summary stats for primary signals:


,HR,RMSSD,SDRR,MEAN_RR
count,11973.000000,11973.000000,11973.000000,11973.000000
mean,69.972471,16.593824,80.000748,873.226568
std,5.526874,3.609901,65.025508,92.413038
min,48.183943,8.888331,40.190464,745.899664
25%,66.302058,14.011592,54.742974,819.513615
50%,70.941799,16.779047,63.880911,850.641348
75%,73.581285,18.769950,77.574982,912.493507
max,81.092551,31.733072,418.066612,1341.611513


In [23]:
baseline_df = df[df['condition'] == 'baseline']

# Central tendency and dispersion
baseline_stats = {}
for col in features:
    med = baseline_df[col].median()
    mad = (baseline_df[col] - med).abs().median()
    baseline_stats[col] = {'median': med, 'mad': mad}

print("=== Layer 1: Personal Digital Baseline (Subject 2) ===")
for feat, stats in baseline_stats.items():
    print(f"{feat:10s} -> Median: {stats['median']:6.2f} | MAD: {stats['mad']:6.2f}")

=== Layer 1: Personal Digital Baseline (Subject 2) ===
HR         -> Median:  71.56 | MAD:   0.90
RMSSD      -> Median:  17.52 | MAD:   0.95
SDRR       -> Median:  58.77 | MAD:   4.55
MEAN_RR    -> Median: 842.85 | MAD:  10.81


In [24]:
# contamination determines how strict we are; 0.05 means we assume 5% of baseline data might be noise
iso_forest = IsolationForest(contamination=0.05, random_state=42)

# Train ONLY on the astronaut's personal baseline
iso_forest.fit(baseline_df[features])

# Predict anomalies across the ENTIRE mission/dataset
# IsolationForest outputs 1 for normal, -1 for anomaly. We'll map it to 0 and 1 for easier processing.
df['anomaly_raw'] = iso_forest.predict(df[features])
df['anomaly_raw'] = df['anomaly_raw'].map({1: 0, -1: 1})

# Let's see how many raw anomalies were detected per condition
print("=== Raw Anomaly Count per Condition ===")
print(df.groupby('condition')['anomaly_raw'].sum())

=== Raw Anomaly Count per Condition ===
condition
amusement     1492
baseline       231
meditation    2865
stress        2431
Name: anomaly_raw, dtype: int64
